# 04 — Transfer Learning
## AT&T Spam Detector

### Objective

This notebook investigates whether transfer learning can improve SMS spam classification by reusing language representations learned from a substantially larger external corpus.

The experiment will:

1. load the same prepared train and validation splits used by previous models;
2. select a pretrained text representation compatible with the current environment;
3. build a binary spam classifier on top of the pretrained representation;
4. train and validate the model without using the test set;
5. evaluate precision, recall, F1-score, ROC-AUC, PR-AUC, and classification errors;
6. compare transfer learning with the baseline and GRU models.

The test set remains reserved for final model evaluation.


In [1]:
# ---------------------------------------------------------------------------
# Environment check for the transfer-learning experiment
# ---------------------------------------------------------------------------

import sys
from importlib.util import find_spec

import numpy as np
import pandas as pd
import tensorflow as tf

RANDOM_STATE = 42

np.random.seed(RANDOM_STATE)
tf.random.set_seed(RANDOM_STATE)

print(f"Python version:     {sys.version.split()[0]}")
print(f"TensorFlow version: {tf.__version__}")
print(f"NumPy version:      {np.__version__}")
print(f"Pandas version:     {pd.__version__}")

print("\nTransfer-learning libraries:")

for package in [
    "tensorflow_hub",
    "tensorflow_text",
    "transformers",
]:
    installed = find_spec(package) is not None
    print(
        f"{package:16s}: "
        f"{'installed' if installed else 'not installed'}"
    )

Python version:     3.12.7
TensorFlow version: 2.21.0
NumPy version:      2.5.2
Pandas version:     3.0.5

Transfer-learning libraries:
tensorflow_hub  : not installed
tensorflow_text : not installed
transformers    : not installed


In [1]:
# ---------------------------------------------------------------------------
# Verify TensorFlow Hub installation
# ---------------------------------------------------------------------------

import tensorflow as tf
import tensorflow_hub as hub

print(f"TensorFlow version:     {tf.__version__}")
print(f"TensorFlow Hub version: {hub.__version__}")

ModuleNotFoundError: No module named 'pkg_resources'

In [1]:
# ---------------------------------------------------------------------------
# Verify TensorFlow Hub after restoring pkg_resources compatibility
# ---------------------------------------------------------------------------

import tensorflow as tf
import tensorflow_hub as hub

print(f"TensorFlow version:     {tf.__version__}")
print(f"TensorFlow Hub version: {hub.__version__}")

c:\Users\Alex\AppData\Local\Programs\Python\Python312\Lib\site-packages\tensorflow_hub\__init__.py:61: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import parse_version



TensorFlow version:     2.21.0
TensorFlow Hub version: 0.16.1


### 4.1 Load the pretrained Universal Sentence Encoder

The Universal Sentence Encoder (USE) is used as a pretrained text representation.

Unlike the previous models, USE does not learn word embeddings from the SMS dataset. Instead, each complete message is mapped directly to a pretrained semantic embedding learned from a much larger external corpus.

Before constructing a classifier, the encoder is tested independently to verify that it loads correctly and produces valid sentence representations.

In [2]:
# ---------------------------------------------------------------------------
# Load the pretrained Universal Sentence Encoder
# ---------------------------------------------------------------------------

USE_URL = "https://tfhub.dev/google/universal-sentence-encoder/4"

use_encoder = hub.load(USE_URL)

print("Universal Sentence Encoder loaded successfully.")

Universal Sentence Encoder loaded successfully.


In [3]:
# ---------------------------------------------------------------------------
# Test the pretrained encoder on representative SMS messages
# ---------------------------------------------------------------------------

sample_messages = [
    "Hey, are we still meeting for lunch today?",
    "Congratulations! You have won a free prize. Call now to claim.",
]

sample_embeddings = use_encoder(sample_messages)

print(f"Embedding tensor shape: {sample_embeddings.shape}")
print(f"Embedding dtype:        {sample_embeddings.dtype}")

Embedding tensor shape: (2, 512)
Embedding dtype:        <dtype: 'float32'>


### 4.2 Load the prepared development splits

The transfer-learning experiment uses the same training and validation observations as the previous models to ensure a fair comparison.

The pretrained encoder will transform raw SMS messages directly into semantic vectors. Unlike the previous word-level models, no project-specific vocabulary or `TextVectorization` layer is required.

The test set remains untouched during model development.

In [5]:
# ---------------------------------------------------------------------------
# Imports and reproducibility
# ---------------------------------------------------------------------------

import sys
from pathlib import Path

import numpy as np
import pandas as pd
import tensorflow as tf
import tensorflow_hub as hub

RANDOM_STATE = 42

np.random.seed(RANDOM_STATE)
tf.random.set_seed(RANDOM_STATE)

print(f"Python version:         {sys.version.split()[0]}")
print(f"TensorFlow version:     {tf.__version__}")
print(f"TensorFlow Hub version: {hub.__version__}")
print(f"NumPy version:          {np.__version__}")
print(f"Pandas version:         {pd.__version__}")

Python version:         3.12.7
TensorFlow version:     2.21.0
TensorFlow Hub version: 0.16.1
NumPy version:          2.5.2
Pandas version:         3.0.5


In [6]:
# ---------------------------------------------------------------------------
# Load the prepared training and validation splits
# ---------------------------------------------------------------------------

from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
PROCESSED_DATA_DIR = PROJECT_ROOT / "data" / "processed"

TRAIN_PATH = PROCESSED_DATA_DIR / "train.csv"
VAL_PATH = PROCESSED_DATA_DIR / "validation.csv"

train_df = pd.read_csv(TRAIN_PATH)
val_df = pd.read_csv(VAL_PATH)

print(f"Train shape:      {train_df.shape}")
print(f"Validation shape: {val_df.shape}")

print("\nTraining class counts:")
print(train_df["label"].value_counts())

print("\nValidation class counts:")
print(val_df["label"].value_counts())

Train shape:      (3610, 2)
Validation shape: (774, 2)

Training class counts:
label
ham     3161
spam     449
Name: count, dtype: int64

Validation class counts:
label
ham     677
spam     97
Name: count, dtype: int64


In [7]:
# ---------------------------------------------------------------------------
# Prepare raw SMS inputs and binary labels
# ---------------------------------------------------------------------------

X_train = train_df["text"].astype(str)
X_val = val_df["text"].astype(str)

label_mapping = {
    "ham": 0,
    "spam": 1,
}

y_train = train_df["label"].map(label_mapping)
y_val = val_df["label"].map(label_mapping)

print(f"Training messages:   {len(X_train)}")
print(f"Validation messages: {len(X_val)}")

print(
    f"Missing training labels: "
    f"{y_train.isna().sum()}"
)

print(
    f"Missing validation labels: "
    f"{y_val.isna().sum()}"
)

Training messages:   3610
Validation messages: 774
Missing training labels: 0
Missing validation labels: 0


### 4.3 Generate pretrained sentence embeddings

The Universal Sentence Encoder transforms each complete SMS into a 512-dimensional dense vector.

These representations are pretrained and remain frozen during this experiment. Only the downstream binary classification layers will be learned from the AT&T SMS dataset.

This makes the experiment a transfer-learning feature-extraction approach.

In [8]:
# ---------------------------------------------------------------------------
# Encode SMS messages with the pretrained Universal Sentence Encoder
# ---------------------------------------------------------------------------

def encode_messages(messages, batch_size=64):
    """
    Encode a collection of SMS messages into frozen 512-dimensional
    Universal Sentence Encoder representations.
    """

    # Convert the messages to a TensorFlow dataset for batched encoding.
    dataset = tf.data.Dataset.from_tensor_slices(
        messages.to_numpy()
    ).batch(batch_size)

    embedding_batches = []

    # Generate pretrained embeddings one batch at a time.
    for batch in dataset:
        batch_embeddings = use_encoder(batch)
        embedding_batches.append(batch_embeddings.numpy())

    # Combine all encoded batches into one matrix.
    return np.concatenate(
        embedding_batches,
        axis=0,
    )


X_train_use = encode_messages(X_train)
X_val_use = encode_messages(X_val)

print(f"Training embeddings shape:   {X_train_use.shape}")
print(f"Validation embeddings shape: {X_val_use.shape}")

print(f"Training dtype:   {X_train_use.dtype}")
print(f"Validation dtype: {X_val_use.dtype}")

Training embeddings shape:   (3610, 512)
Validation embeddings shape: (774, 512)
Training dtype:   float32
Validation dtype: float32


In [9]:
# ---------------------------------------------------------------------------
# Verify integrity of the pretrained representations
# ---------------------------------------------------------------------------

print(
    "Training embeddings contain NaN:",
    np.isnan(X_train_use).any(),
)

print(
    "Validation embeddings contain NaN:",
    np.isnan(X_val_use).any(),
)

print(
    "Training embeddings contain infinity:",
    np.isinf(X_train_use).any(),
)

print(
    "Validation embeddings contain infinity:",
    np.isinf(X_val_use).any(),
)

Training embeddings contain NaN: False
Validation embeddings contain NaN: False
Training embeddings contain infinity: False
Validation embeddings contain infinity: False


### 4.4 Build a classifier on top of frozen USE embeddings

The Universal Sentence Encoder remains frozen and is used only to generate pretrained 512-dimensional representations.

A small feed-forward neural network is trained on top of these embeddings to perform binary spam classification.

This separates the pretrained language representation from the task-specific classifier and provides a clean transfer-learning feature-extraction experiment.

In [10]:
# ---------------------------------------------------------------------------
# Build the classifier on top of frozen USE embeddings
# ---------------------------------------------------------------------------

from tensorflow.keras import Sequential
from tensorflow.keras.layers import Dense, Dropout, Input

use_classifier = Sequential(
    [
        # Accept one pretrained 512-dimensional USE vector per SMS.
        Input(
            shape=(512,),
            name="use_embedding",
        ),

        # Learn task-specific nonlinear combinations of the pretrained features.
        Dense(
            128,
            activation="relu",
            name="dense_hidden_1",
        ),

        # Reduce overfitting on the relatively small SMS dataset.
        Dropout(
            0.3,
            name="dropout_1",
        ),

        # Compress the representation before binary classification.
        Dense(
            32,
            activation="relu",
            name="dense_hidden_2",
        ),

        Dropout(
            0.2,
            name="dropout_2",
        ),

        # Output the probability that the SMS belongs to the spam class.
        Dense(
            1,
            activation="sigmoid",
            name="spam_probability",
        ),
    ],
    name="use_transfer_classifier",
)

use_classifier.summary()

Model: "use_transfer_classifier"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_hidden_1 (Dense)          │ (None, 128)            │        65,664 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_hidden_2 (Dense)          │ (None, 32)             │         4,128 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ spam_probability (Dense)        │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 69,825 (272.75 KB)

 Trainable params: 69,825 (272.75 KB)

 Non-trainable params: 0 (0.00 B)

### 4.5 Compile the transfer-learning classifier

The task-specific classifier is trained using the same binary cross-entropy objective and evaluation metrics as the previous models.

Keeping the evaluation setup consistent allows the USE-based model to be compared fairly with the baseline and GRU architectures.

In [11]:
# ---------------------------------------------------------------------------
# Compile the USE-based classifier
# ---------------------------------------------------------------------------

from tensorflow.keras.metrics import (
    BinaryAccuracy,
    Precision,
    Recall,
    AUC,
)

use_classifier.compile(
    # Adam is used consistently across the neural-network experiments.
    optimizer="adam",

    # Binary cross-entropy is appropriate for the binary spam/ham target.
    loss="binary_crossentropy",

    # Match the previous models for direct validation comparison.
    metrics=[
        BinaryAccuracy(name="accuracy"),
        Precision(name="precision"),
        Recall(name="recall"),
        AUC(name="roc_auc"),
        AUC(name="pr_auc", curve="PR"),
    ],
)

In [12]:
# ---------------------------------------------------------------------------
# Configure early stopping and learning-rate reduction
# ---------------------------------------------------------------------------

from tensorflow.keras.callbacks import (
    EarlyStopping,
    ReduceLROnPlateau,
)

early_stopping_use = EarlyStopping(
    monitor="val_loss",
    patience=3,
    restore_best_weights=True,
)

reduce_lr_use = ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.5,
    patience=2,
    min_lr=1e-6,
)

### 4.6 Train the USE-based classifier

Only the task-specific dense classification head is trained.

The pretrained USE representations remain fixed throughout training, so the experiment measures how effectively pretrained semantic information transfers to SMS spam detection.

The same training and validation splits are used as in the previous experiments, and the test set remains untouched.

In [13]:
# ---------------------------------------------------------------------------
# Train the classifier on frozen USE representations
# ---------------------------------------------------------------------------

EPOCHS = 20
BATCH_SIZE = 32

history_use = use_classifier.fit(
    X_train_use,
    y_train,

    validation_data=(
        X_val_use,
        y_val,
    ),

    batch_size=BATCH_SIZE,
    epochs=EPOCHS,

    callbacks=[
        early_stopping_use,
        reduce_lr_use,
    ],

    verbose=1,
)

Epoch 1/20
113/113 ━━━━━━━━━━━━━━━━━━━━ 3s 10ms/step - accuracy: 0.9235 - loss: 0.2361 - pr_auc: 0.7071 - precision: 0.9023 - recall: 0.4321 - roc_auc: 0.9245 - val_accuracy: 0.9832 - val_loss: 0.0582 - val_pr_auc: 0.9808 - val_precision: 0.9468 - val_recall: 0.9175 - val_roc_auc: 0.9970 - learning_rate: 0.0010
Epoch 2/20
113/113 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - accuracy: 0.9825 - loss: 0.0578 - pr_auc: 0.9690 - precision: 0.9468 - recall: 0.9109 - roc_auc: 0.9927 - val_accuracy: 0.9832 - val_loss: 0.0428 - val_pr_auc: 0.9863 - val_precision: 0.9565 - val_recall: 0.9072 - val_roc_auc: 0.9979 - learning_rate: 0.0010
Epoch 3/20
113/113 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - accuracy: 0.9870 - loss: 0.0414 - pr_auc: 0.9832 - precision: 0.9568 - recall: 0.9376 - roc_auc: 0.9972 - val_accuracy: 0.9832 - val_loss: 0.0401 - val_pr_auc: 0.9871 - val_precision: 0.9667 - val_recall: 0.8969 - val_roc_auc: 0.9980 - learning_rate: 0.0010
Epoch 4/20
113/113 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - accuracy:

### 4.7 Evaluate the restored USE-based classifier

The USE-based classifier achieves its lowest validation loss before the final training epoch.

Because early stopping restores the weights associated with the best validation loss, the retained classifier is evaluated separately after training.

This provides the metrics that will be used for direct comparison with the baseline and GRU models.

In [14]:
# ---------------------------------------------------------------------------
# Evaluate the restored USE classifier on validation data
# ---------------------------------------------------------------------------

validation_results_use = use_classifier.evaluate(
    X_val_use,
    y_val,
    verbose=0,
    return_dict=True,
)

print("USE validation metrics using restored best weights:\n")

for metric_name, metric_value in validation_results_use.items():
    print(f"{metric_name:10s}: {metric_value:.4f}")

USE validation metrics using restored best weights:

accuracy  : 0.9858
loss      : 0.0367
pr_auc    : 0.9883
precision : 0.9479
recall    : 0.9381
roc_auc   : 0.9982


In [15]:
# ---------------------------------------------------------------------------
# Generate validation predictions at the default threshold
# ---------------------------------------------------------------------------

from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    precision_score,
    recall_score,
    f1_score,
)

# Generate continuous spam probabilities.
val_probabilities_use = use_classifier.predict(
    X_val_use,
    verbose=0,
).ravel()

# Apply the default binary classification threshold.
val_predictions_use_05 = (
    val_probabilities_use >= 0.50
).astype(int)

precision_use_05 = precision_score(
    y_val,
    val_predictions_use_05,
)

recall_use_05 = recall_score(
    y_val,
    val_predictions_use_05,
)

f1_use_05 = f1_score(
    y_val,
    val_predictions_use_05,
)

print(f"Precision: {precision_use_05:.4f}")
print(f"Recall:    {recall_use_05:.4f}")
print(f"F1-score:  {f1_use_05:.4f}")

print("\nClassification report:")
print(
    classification_report(
        y_val,
        val_predictions_use_05,
        target_names=["ham", "spam"],
        digits=4,
    )
)

print("Confusion matrix:")
print(
    confusion_matrix(
        y_val,
        val_predictions_use_05,
    )
)

Precision: 0.9479
Recall:    0.9381
F1-score:  0.9430

Classification report:
              precision    recall  f1-score   support

         ham     0.9912    0.9926    0.9919       677
        spam     0.9479    0.9381    0.9430        97

    accuracy                         0.9858       774
   macro avg     0.9695    0.9654    0.9674       774
weighted avg     0.9857    0.9858    0.9858       774

Confusion matrix:
[[672   5]
 [  6  91]]


### 4.8 Investigate the USE classification threshold

The USE-based classifier may produce probability scores with a different calibration from the previous models.

Its operating threshold is therefore selected independently using validation data only.

The test set remains untouched.

In [16]:
# ---------------------------------------------------------------------------
# Compare candidate thresholds on validation data
# ---------------------------------------------------------------------------

threshold_results_use = []

candidate_thresholds = np.arange(
    0.10,
    0.91,
    0.05,
)

for threshold in candidate_thresholds:

    predictions = (
        val_probabilities_use >= threshold
    ).astype(int)

    tn, fp, fn, tp = confusion_matrix(
        y_val,
        predictions,
    ).ravel()

    threshold_results_use.append({
        "threshold": threshold,
        "precision": precision_score(
            y_val,
            predictions,
            zero_division=0,
        ),
        "recall": recall_score(
            y_val,
            predictions,
            zero_division=0,
        ),
        "f1": f1_score(
            y_val,
            predictions,
            zero_division=0,
        ),
        "false_positives": fp,
        "false_negatives": fn,
    })

threshold_df_use = pd.DataFrame(
    threshold_results_use
)

print(
    threshold_df_use.round(4).to_string(
        index=False
    )
)

 threshold  precision  recall     f1  false_positives  false_negatives
      0.10     0.8421  0.9897 0.9100               18                1
      0.15     0.8482  0.9794 0.9091               17                2
      0.20     0.8679  0.9485 0.9064               14                5
      0.25     0.8762  0.9485 0.9109               13                5
      0.30     0.9020  0.9485 0.9246               10                5
      0.35     0.9200  0.9485 0.9340                8                5
      0.40     0.9293  0.9485 0.9388                7                5
      0.45     0.9381  0.9381 0.9381                6                6
      0.50     0.9479  0.9381 0.9430                5                6
      0.55     0.9474  0.9278 0.9375                5                7
      0.60     0.9667  0.8969 0.9305                3               10
      0.65     0.9667  0.8969 0.9305                3               10
      0.70     0.9667  0.8969 0.9305                3               10
      

In [17]:
# ---------------------------------------------------------------------------
# Identify the threshold with the highest validation F1-score
# ---------------------------------------------------------------------------

best_threshold_row_use = threshold_df_use.loc[
    threshold_df_use["f1"].idxmax()
]

print("Best USE threshold by validation F1-score:")
print(
    best_threshold_row_use.round(4)
)

Best USE threshold by validation F1-score:
threshold          0.5000
precision          0.9479
recall             0.9381
f1                 0.9430
false_positives    5.0000
false_negatives    6.0000
Name: 8, dtype: float64


### 4.9 Compare the three modeling approaches

The transfer-learning model is compared with the baseline and GRU experiments using their validation-selected operating thresholds.

The comparison considers threshold-dependent classification performance, threshold-independent ranking metrics, and the balance between false positives and false negatives.

In [18]:
# ---------------------------------------------------------------------------
# Compare validation performance across all three models
# ---------------------------------------------------------------------------

model_comparison = pd.DataFrame(
    [
        {
            "model": "Embedding + GlobalAveragePooling1D",
            "threshold": 0.40,
            "precision": 0.8932,
            "recall": 0.9485,
            "f1": 0.9200,
            "roc_auc": 0.9939,
            "pr_auc": 0.9531,
            "false_positives": 11,
            "false_negatives": 5,
        },
        {
            "model": "Embedding + GRU",
            "threshold": 0.89,
            "precision": 1.0000,
            "recall": 0.8866,
            "f1": 0.9399,
            "roc_auc": 0.9847,
            "pr_auc": 0.9640,
            "false_positives": 0,
            "false_negatives": 11,
        },
        {
            "model": "Universal Sentence Encoder",
            "threshold": 0.50,
            "precision": precision_use_05,
            "recall": recall_use_05,
            "f1": f1_use_05,
            "roc_auc": validation_results_use["roc_auc"],
            "pr_auc": validation_results_use["pr_auc"],
            "false_positives": 5,
            "false_negatives": 6,
        },
    ]
)

model_comparison.round(4)

,model,threshold,precision,recall,f1,roc_auc,pr_auc,false_positives,false_negatives
0,Embedding + GlobalAveragePooling1D,0.40,0.8932,0.9485,0.9200,0.9939,0.9531,11,5
1,Embedding + GRU,0.89,1.0000,0.8866,0.9399,0.9847,0.9640,0,11
2,Universal Sentence Encoder,0.50,0.9479,0.9381,0.9430,0.9982,0.9883,5,6


#### Interim model-selection interpretation

The Universal Sentence Encoder provides the strongest overall validation performance.

At its selected threshold of 0.50, USE achieves the highest F1-score (0.9430), ROC-AUC (0.9982), and PR-AUC (0.9883) of the three models. It also maintains a balanced operating profile, with 5 false positives and 6 false negatives.

The baseline retains the highest spam recall (0.9485), missing only 5 spam messages, but produces substantially more false positives and a lower F1-score.

The GRU achieves perfect validation precision at its selected threshold and eliminates false positives, but its lower recall causes 11 spam messages to be missed. Its additional sequential complexity therefore does not translate into the strongest overall validation performance.

USE appears to benefit from pretrained semantic representations learned from a substantially larger external corpus. This is particularly valuable for the relatively small SMS dataset, where models learning their representations entirely from scratch have less linguistic information available.

Based on validation performance, USE is currently the leading candidate for final evaluation. However, the test set remains untouched until model development and error analysis are complete.

### 4.10 USE error analysis

The false-positive and false-negative predictions produced by the USE classifier are inspected to understand the remaining failure modes.

This analysis is particularly useful for determining whether pretrained semantic representations resolve errors observed in the models trained from scratch.

In [19]:
# ---------------------------------------------------------------------------
# Build an error-analysis table at the selected USE threshold
# ---------------------------------------------------------------------------

USE_THRESHOLD = 0.50

val_predictions_use = (
    val_probabilities_use >= USE_THRESHOLD
).astype(int)

error_analysis_use = val_df.copy()

error_analysis_use["true_label"] = y_val.to_numpy()
error_analysis_use["spam_probability"] = val_probabilities_use
error_analysis_use["prediction"] = val_predictions_use

false_positives_use = error_analysis_use[
    (error_analysis_use["true_label"] == 0)
    & (error_analysis_use["prediction"] == 1)
].copy()

false_negatives_use = error_analysis_use[
    (error_analysis_use["true_label"] == 1)
    & (error_analysis_use["prediction"] == 0)
].copy()

print(
    f"False positives: {len(false_positives_use)}"
)

print(
    f"False negatives: {len(false_negatives_use)}"
)

False positives: 5
False negatives: 6


In [20]:
# ---------------------------------------------------------------------------
# Inspect legitimate SMS messages incorrectly classified as spam
# ---------------------------------------------------------------------------

false_positives_use[
    [
        "text",
        "spam_probability",
    ]
].sort_values(
    "spam_probability",
    ascending=False,
)

,text,spam_probability
577,"Brainless Baby Doll..:-D;-), vehicle sariyag d...",0.878758
481,"Hi Shanil,Rakhesh here.thanks,i have exchanged...",0.763988
609,"\NONE!NOWHERE IKNO DOESDISCOUNT!SHITINNIT\""""",0.724998
353,As per your request 'Melle Melle (Oru Minnamin...,0.564611
352,Hi.:)technical support.providing assistance to...,0.563276


In [21]:
# ---------------------------------------------------------------------------
# Inspect spam SMS messages incorrectly classified as legitimate
# ---------------------------------------------------------------------------

false_negatives_use[
    [
        "text",
        "spam_probability",
    ]
].sort_values(
    "spam_probability",
    ascending=True,
)

,text,spam_probability
219,dating:i have had two of these. Only started a...,0.051995
667,"SMS. ac JSco: Energy is high, but u may not kn...",0.128669
716,Monthly password for wap. mobsi.com is 391784....,0.166046
411,For sale - arsenal dartboard. Good condition b...,0.176169
726,Babe: U want me dont u baby! Im nasty and have...,0.186279
597,We have new local dates in your area - Lots of...,0.438147


#### USE error interpretation

The USE classifier produces 5 false positives and 6 false negatives at its selected threshold of 0.50.

The false positives frequently contain characteristics that can plausibly resemble unsolicited or commercial communication, including service-oriented language, unusual punctuation, formulaic wording, and transactional content. These examples illustrate the semantic ambiguity between legitimate informational messages and spam.

Several false negatives overlap with difficult cases observed in the models trained from scratch. Examples include atypically phrased promotional, dating, subscription, and sales-related messages. Some receive relatively low spam probabilities, indicating that the remaining errors cannot be resolved simply through a small adjustment of the decision threshold.

One false negative lies closer to the decision boundary, but lowering the threshold would also increase the number of legitimate messages classified as spam. The validation threshold analysis therefore supports retaining the default threshold of 0.50.

Overall, pretrained sentence representations improve the balance between spam detection and false alarms, but they do not eliminate ambiguity in short, informal SMS language.

In [22]:
# ---------------------------------------------------------------------------
# Calculate and save exact USE validation metrics
# ---------------------------------------------------------------------------

final_use_predictions = (
    val_probabilities_use >= USE_THRESHOLD
).astype(int)

final_use_precision = precision_score(
    y_val,
    final_use_predictions,
    zero_division=0,
)

final_use_recall = recall_score(
    y_val,
    final_use_predictions,
    zero_division=0,
)

final_use_f1 = f1_score(
    y_val,
    final_use_predictions,
    zero_division=0,
)

final_use_confusion_matrix = confusion_matrix(
    y_val,
    final_use_predictions,
)

use_validation_metrics = pd.DataFrame(
    [
        {
            "model": "Universal Sentence Encoder",
            "selected_threshold": USE_THRESHOLD,
            "validation_accuracy": validation_results_use["accuracy"],
            "validation_precision": final_use_precision,
            "validation_recall": final_use_recall,
            "validation_f1": final_use_f1,
            "validation_roc_auc": validation_results_use["roc_auc"],
            "validation_pr_auc": validation_results_use["pr_auc"],
            "validation_false_positives": int(
                final_use_confusion_matrix[0, 1]
            ),
            "validation_false_negatives": int(
                final_use_confusion_matrix[1, 0]
            ),
        }
    ]
)

METRICS_DIR = PROJECT_ROOT / "outputs" / "metrics"
METRICS_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

USE_METRICS_PATH = (
    METRICS_DIR
    / "use_validation_metrics.csv"
)

use_validation_metrics.to_csv(
    USE_METRICS_PATH,
    index=False,
)

print(use_validation_metrics.round(4))
print(f"\nMetrics saved to: {USE_METRICS_PATH}")

                        model  selected_threshold  validation_accuracy  \
0  Universal Sentence Encoder                 0.5               0.9858   

   validation_precision  validation_recall  validation_f1  validation_roc_auc  \
0                0.9479             0.9381          0.943              0.9982   

   validation_pr_auc  validation_false_positives  validation_false_negatives  
0             0.9883                           5                           6  

Metrics saved to: c:\Users\Alex\Desktop\Jehda AI\Fullstack\CDSD Certification\4_Att-spam-detector\outputs\metrics\use_validation_metrics.csv


In [23]:
# ---------------------------------------------------------------------------
# Save the trained task-specific USE classifier
# ---------------------------------------------------------------------------

MODELS_DIR = PROJECT_ROOT / "models"
MODELS_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

USE_CLASSIFIER_PATH = (
    MODELS_DIR
    / "use_transfer_classifier.keras"
)

use_classifier.save(
    USE_CLASSIFIER_PATH
)

print(
    f"Classifier saved to: {USE_CLASSIFIER_PATH}"
)

Classifier saved to: c:\Users\Alex\Desktop\Jehda AI\Fullstack\CDSD Certification\4_Att-spam-detector\models\use_transfer_classifier.keras


## 4.13 Conclusion

Transfer learning was evaluated using the Universal Sentence Encoder (USE) as a frozen pretrained feature extractor. Each raw SMS message was transformed into a 512-dimensional semantic representation, and a small task-specific neural-network classifier was trained on top.

The classifier contained only 69,825 trainable parameters because the pretrained USE representation itself was not fine-tuned.

The lowest validation loss was obtained at epoch 5. After this point, training loss continued to decrease while validation loss stopped improving, indicating the beginning of overfitting. Early stopping restored the best weights.

At the default threshold of 0.50, the USE classifier achieved:

- validation accuracy: **0.9858**
- precision: **0.9479**
- recall: **0.9381**
- F1-score: **0.9430**
- ROC-AUC: **0.9982**
- PR-AUC: **0.9883**
- false positives: **5**
- false negatives: **6**

Validation threshold analysis found that 0.50 also produced the highest observed F1-score, so no alternative operating threshold was required.

Compared with the previous experiments, USE provides the strongest overall validation performance. It achieves the highest F1-score, ROC-AUC, and PR-AUC while maintaining a balanced precision-recall profile. The baseline retains slightly higher spam recall, while the GRU achieves higher precision at the cost of substantially lower recall.

Error analysis shows that pretrained semantic representations improve classification but do not eliminate ambiguity in atypically phrased promotional, dating, subscription, and service-oriented SMS messages.

Based on validation performance, the USE-based classifier is selected as the leading candidate for final evaluation.

The test set has remained untouched during model development. It will be evaluated only after the development-stage model and decision threshold have been frozen.

In [24]:
# ---------------------------------------------------------------------------
# Verify that the trained classifier artifact was saved successfully
# ---------------------------------------------------------------------------

print(f"Classifier path: {USE_CLASSIFIER_PATH}")
print(f"Classifier exists: {USE_CLASSIFIER_PATH.exists()}")

Classifier path: c:\Users\Alex\Desktop\Jehda AI\Fullstack\CDSD Certification\4_Att-spam-detector\models\use_transfer_classifier.keras
Classifier exists: True
